In [2]:
!pip install sdv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.5/159.5 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.9/139.9 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 83.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.5/52.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.4/193.4 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.8/84.8 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 102.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 80.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8

In [3]:
# 제출 파일 생성 관련
import os
import zipfile

# 데이터 처리 및 분석
import pandas as pd
import numpy as np
from scipy import stats
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt

# 머신러닝 전처리
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder

# 머신러닝 모델
import xgboost as xgb

# 합성 데이터 생성
from sdv.metadata import SingleTableMetadata
from sdv.single_table import CTGANSynthesizer

# To ignore all warnings
import warnings
warnings.filterwarnings('ignore')

In [4]:
train_all = pd.read_csv("./data/train_kor.csv") # 경로는 각자 지정
test_all = pd.read_csv("./data/test_kor.csv") # 경로는 각자 지정

In [5]:
train = train_all.drop(columns="샘플 식별자 번호")

In [6]:
'''
(*) 리더보드 산식 중 생성데이터의 익명성(TCAP)채점을 위해 각 클래스 별로 1000개의 생성데이터가 반드시 필요합니다.
(*) 본 베이스 라인에서는 "계좌의 거래 재개 일자" 13종류에 대해 1000개씩 , 총 13,000개의 데이터를 생성할 예정입니다.
(*) 분류 모델 성능 개선을 위해 생성 데이터를 활용하는 것에는 생성 데이터의 Row 개수에 제한이 없습니다. 단, 리더보드 평가를 위해 제출을 하는 생성 데이터 프레임은 익명성(TCAP) 평가를 위함이며, 위의 조건을 갖춘 생성 데이터를 제출해야합니다.
'''
N_CLS_PER_GEN = 1000


In [7]:

# 이상치 처리 함수
def handle_outliers(series, n_std=3):
    mean = series.mean()
    std = series.std()
    z_scores = np.abs(stats.zscore(series))
    return series.mask(z_scores > n_std, mean)

# Time_difference 컬럼을 총 초로 변환 및 이상치 처리
train['거래시간_차이_seconds'] = pd.to_timedelta(train['직전 거래와의 시간 차이']).dt.total_seconds()
train['거래시간_차이_seconds'] = handle_outliers(train['거래시간_차이_seconds'])


# 모든 Fraud_Type 목록 생성 (m 포함)
fraud_types = train['사기 시나리오 (예측 목표)'].unique()

# 모든 합성 데이터를 저장할 DataFrame 초기화
all_synthetic_data = pd.DataFrame()

N_SAMPLE = 100

# 각 Fraud_Type에 대해 합성 데이터 생성 및 저장
for fraud_type in tqdm(fraud_types):

    # 해당 Fraud_Type에 대한 서브셋 생성
    subset = train[train["사기 시나리오 (예측 목표)"] == fraud_type]

    # 모든 Fraud_Type에 대해 100개씩 샘플링
    subset = subset.sample(n=N_SAMPLE, random_state=42)

    # Time_difference 열 제외 (초 단위로 변환된 컬럼만 사용)
    subset = subset.drop('직전 거래와의 시간 차이', axis=1)

    # 메타데이터 생성 및 모델 학습
    metadata = SingleTableMetadata()

    metadata.detect_from_dataframe(subset)
    metadata.set_primary_key(None)

    # 데이터 타입 설정
    column_sdtypes = {
        '거래 전 잔액': 'numerical',
        '거래 후 잔액': 'numerical',
        '주민번호': 'categorical',
        '고객명': 'categorical',
        '암호화된 계좌번호': 'categorical',
        '거래에 사용한 단말기 IP주소': 'ipv4_address',
        '거래 발생 위치': 'categorical',
        '수취인 계좌번호': 'categorical',
        '계좌의 거래 재개 일자': 'categorical',
        '거래시간_차이_seconds': 'numerical',
        '고객 출생년도': 'numerical'
    }

    # 각 컬럼에 대해 데이터 타입 설정
    for column, sdtype in column_sdtypes.items():
        metadata.update_column(
            column_name=column,
            sdtype=sdtype
        )

    synthesizer = CTGANSynthesizer(
                            metadata,
                            epochs=100
                        )
    synthesizer.fit(subset)

    synthetic_subset = synthesizer.sample(num_rows=N_CLS_PER_GEN)

    # 생성된 Time_difference_seconds의 이상치 처리
    synthetic_subset['거래시간_차이_seconds'] = handle_outliers(synthetic_subset['거래시간_차이_seconds'])

    # Time_difference_seconds를 다시 timedelta로 변환
    synthetic_subset['직전 거래와의 시간 차이'] = pd.to_timedelta(synthetic_subset['거래시간_차이_seconds'], unit='s')

    # Time_difference_seconds 컬럼 제거
    synthetic_subset = synthetic_subset.drop('거래시간_차이_seconds', axis=1)

    # 생성된 데이터를 all_synthetic_data에 추가
    all_synthetic_data = pd.concat([all_synthetic_data, synthetic_subset], ignore_index=True)
# 최종 결과 확인
print("\nFinal All Synthetic Data Shape:", all_synthetic_data.shape)

100%|██████████| 13/13 [21:09<00:00, 97.65s/it]


Final All Synthetic Data Shape: (13000, 63)


In [8]:
all_synthetic_data.head() # 잘 변환되었는지 확인하는 과정입니다.

,고객 출생년도,고객 성별,고객명,주민번호,고객 등록일자,고객 등급,3개월 이내 금융/공동인증서 발급 여부,3개월 이내 사설인증서 발급 여부,3개월 이내 보안카드 및 OTP 발급 여부,3개월 이내 개인정보 수정 여부,...,마지막 영업점 거래 일자,7일 거래내역 중 1천만원 이상 입금 여부,7일 거래내역 중 미거래 계좌 여부,수취계좌의 거래중지계좌 해당 여부,3시간 이내 해당 수취계좌에 이체 횟수,해당 수취계좌와 거래한 횟수,60세 이후 iOS 첫 사용자,사기 시나리오 (예측 목표),계좌의 거래 재개 일자,직전 거래와의 시간 차이
0,1966,female,김영호,dhITBu-DbhAPPn,2012-12-10 22:02:43,D,0,1,1,0,...,2019-09-13 00:21:18,0,0,0,0,0,0,m,2030-06-03 14:14:54,0 days 00:01:38
1,1980,male,이재현,VUdWiC-wXhKmwF,2007-03-25 00:29:47,A,1,1,0,1,...,2011-05-23 14:22:38,0,1,1,1,0,0,m,2030-06-03 14:14:54,19 days 10:54:21
2,1991,female,최유진,UWTuyh-pgXndzV,2012-12-10 22:02:43,A,1,1,1,1,...,2010-07-08 02:23:35,0,1,0,0,0,0,m,2014-05-20 21:31:48,2 days 18:41:06
3,1988,male,이민재,BDBAtF-ZmBUHYl,2006-11-24 04:54:51,D,1,1,0,1,...,2004-08-05 00:53:58,0,0,1,0,0,0,m,2039-02-06 11:20:56,2 days 00:06:39
4,1999,female,김영환,UyFsAc-KBfHGXe,2007-12-20 06:37:08,B,1,1,1,0,...,2020-08-07 17:10:02,0,0,0,2,0,0,m,2004-07-10 20:11:30,12 days 20:26:11.086000


In [9]:
origin_train = train_all.drop(columns="샘플 식별자 번호")
train_total = pd.concat([origin_train, all_synthetic_data])
train_total.shape

(133000, 63)

In [43]:
train_total['사기 시나리오 (예측 목표)'].value_counts()

,count
사기 시나리오 (예측 목표),
m,119800
a,1100
j,1100
h,1100
k,1100
c,1100
g,1100
i,1100
b,1100


In [10]:
train_x = train_total.drop(columns=['사기 시나리오 (예측 목표)'])
train_y = train_total['사기 시나리오 (예측 목표)']

test_x = test_all.drop(columns=['샘플 식별자 번호'])

In [11]:
def add_custom_features(df):
    df = df.copy()
    current_year = 2024

    # 📌 1. 고객 나이
    df['고객 나이'] = current_year - df['고객 출생년도']

    # 📌 2. 거리 관련 변수
    df['Is_Large_Distance'] = (df['직전 거래 발생지와의 거리 차이'] >= 300).astype(int)

    # 📌 3. A - VPN 로그인과 거리
    vpn_login_types = ['a', 'b']
    df['A_VPN_and_Large_Distance'] = (
        df['거래 시스템 접근 매체(a: ID/PW 로그인, b: 패턴, c: 생체로그인, d: 금융/공동 인증서, e: 사설인증서, f: 보안카드, g: OTP, h: 보안카드+OTP)'].isin(vpn_login_types) &
        (df['Is_Large_Distance'] == 1)
    ).astype(int)

    # 📌 4. B - 고객등급 + 단말 이상
    df['B_Flag_GradeC_DeviceAnomaly'] = (
        (df['고객 등급'] == 'C') &
        ((df['모바일 로밍 여부'] == 1) | (df['탈옥 및 루팅 여부'] == 1) | (df['VPN 사용 여부'] == 1))
    ).astype(int)

    # 📌 5. C - 거래 한도 적고 채널 정상
    df['C_Limit_Under_Max_And_Channel_Not_Others'] = (
        (df['1일 거래 한도'] <= 2000000) & (df['거래 채널'] != 'Others')
    ).astype(int)

    # 📌 6. D - 스크립트 유사 거래
    df['D_Scripted_Like_Transaction'] = (
        (df['7일 거래내역 중 미사용 단말 여부'] == 1) &
        (df['거래 시스템 접근 매체(a: ID/PW 로그인, b: 패턴, c: 생체로그인, d: 금융/공동 인증서, e: 사설인증서, f: 보안카드, g: OTP, h: 보안카드+OTP)'].isin(vpn_login_types)) &
        (df['거래 성공/실패 여부'] == 0) &
        (df['에러코드(a: 에러없음, b: 시스템 오류, c: 잔액부족, d: 이체한도초과, e: 계좌정보 오류, f: 계좌이체 거부)'] == 'a')
    ).astype(int)

    # 📌 7. E - 비대면입금 + ATM출금 위험
    df['E_Vishing_Withdrawal_Risk'] = (
        (df['거래 채널'] == 'ATM') &
        (df['7일 거래내역 중 1천만원 이상 입금 여부'] == 1)
    ).astype(int)

    # 📌 8. F - 머니론더링 (기타 단말 + 기타 채널)
    df['F_MoneyLaundering_Risk'] = (
        (df['거래 채널'] == 'Others') &
        (df['거래에 사용한 단말기 OS'].isin(['Others', 'Windows']))
    ).astype(int)

    # 📌 9. G - 고액입금 + 정지계좌 해제
    df['G_MoneyLaundering_Risk'] = (
        (df['7일 거래내역 중 1천만원 이상 입금 여부'] == 1) &
        (df['30일 이내 계좌 정지 해제 여부'] == 1)
    ).astype(int)

    # 📌 10. H - 한도 높고 과거 이체 작음
    df['H_Flag_HighLimit_LowMonthlyMax'] = (
        (df['1일 거래 한도 잔여액'] > 3000000) &
        (df['1개월 거래내역 중 최대 이체(출금) 금액'] <= 10000000)
    ).astype(int)

    # 📌 11. I - 실패 거래 + 큰 금액 이체 시도
    df['I_Flag_AType_HighMidnightTransfer'] = (
        (df['거래 성공/실패 여부'] == 0) &
        (df['이체 금액'] <= -2500000)
    ).astype(int)

    # 📌 12. J - 중지계좌 + 미거래 계좌
    df['J_Suspicious_Small_Repeated_Transfer'] = (
        (df['수취계좌의 거래중지계좌 해당 여부'] == 1) &
        (df['7일 거래내역 중 미거래 계좌 여부'] == 1)
    ).astype(int)

    # 📌 13. K - 수취계좌 반복 이체
    df['K_Affiliate_Scam_Risk'] = (
        (df['3시간 이내 해당 수취계좌에 이체 횟수'] >= 2) &
        (df['해당 수취계좌와 거래한 횟수'] >= 2)
    ).astype(int)

    # 📌 14. L - 60세 이상 + 신용/담보대출 신청
    df['L_Is_Elderly_Secured_Loan'] = (
        (df['고객 나이'] >= 60) &
        (df['대출 신청 유형(a: 없음, b: 신용대출, c: 담보대출, d: 할부금융, e: 기타)'].isin(['b', 'c']))
    ).astype(int)

    # 📌 15. 급격한 잔액 감소 여부
    df['Is_Sudden_Balance_Drop'] = (
        df['거래 후 잔액'] < df['거래 전 잔액'] * 0.3
    ).astype(int)

    # 📌 16~17. 비율 기반 파생변수
    df['Amount_Per_Limit_Ratio'] = df['이체 금액'] / (df['1일 거래 한도'] + 1)
    df['Amount_vs_Monthly_Max'] = df['이체 금액'] / (df['1개월 거래내역 중 최대 이체(출금) 금액'] + 1)

    # [2] 출금 후 남은 잔액이 기존 금액의 10% 이하
    df['거래금액'] = df['거래 전 잔액'] - df['거래 후 잔액']

    df['잔액 변화 비율'] = df.apply(
        lambda row: row['거래금액'] / row['거래 전 잔액']
        if row['거래 전 잔액'] > 0 else 0,
        axis=1
    )

    # 변화 비율이 90% 이상이면, 잔액이 10% 이하로 줄어든 것
    df['F-잔액이 크게 감소'] = (df['잔액 변화 비율'] >= 0.9).astype(int)
    df.drop(columns = '잔액 변화 비율')
######################################################################################################
        # <1번> 모바일 보안 플래그
    security_cols = [
        '탈옥 및 루팅 여부',
        '모바일 로밍 여부',
        'VPN 사용 여부'
    ]
    security_flag_cnt = df[security_cols].sum(axis=1)
    df['B-모바일 보안_2plus'] = (security_flag_cnt >= 2).astype(int)

    # ────────────────────────────
    # <2번> 터미널 악의적 행동 플래그
    terminal_cols = [
        '전화번호 조작 여부',
        '원격제어 여부',
        '템퍼링 여부',
        '피싱 여부',
        '신뢰할 수 없는 인증서 사용 여부',
        '키로깅 여부'
    ]
    terminal_flag_cnt = df[terminal_cols].sum(axis=1)
    df['C-terminal_malicious_2plus'] = (terminal_flag_cnt >= 2).astype(int)


    security_cols = [
        '전화번호 조작 여부',
        '모바일 로밍 여부',
        'VPN 사용 여부',
        '템퍼링 여부',
        '피싱 여부',
        '신뢰할 수 없는 인증서 사용 여부'
    ]

    # 조건: 값이 0인 컬럼 개수를 세기
    num_zeros = (df[security_cols] == 0).sum(axis=1)

    # 2개 이상 0인 경우 → 1, 아니면 0
    df['E,F-탐지_2plus'] = (num_zeros >= 6).astype(int)

    # ✅ [2] 추가 보안 조합 변수
    other_security_cols = ['탈옥 및 루팅 여부', '모바일 로밍 여부', '키로깅 여부']
    df['D-two_or_more_security_flags_zero'] = (
        (df[other_security_cols] == 0).sum(axis=1) >= 2
    ).astype(int)

    # 조합 파생변수
    df['E,F-정지해제이후_입금1000'] = (
        (df['30일 이내 계좌 정지 해제 여부'] == 1) &
        (df['7일 거래내역 중 1천만원 이상 입금 여부'] == 1)
    ).astype(int)

    df['E,F-해제이후_단말기_기타or윈도우'] = (
        (df['30일 이내 계좌 정지 해제 여부'] == 1) &
        (df['거래에 사용한 단말기 OS'].isin(['Others', 'Window']))
    ).astype(int)

    df['E,F-해제이후_채널_기타'] = (
        (df['30일 이내 계좌 정지 해제 여부'] == 1) &
        (df['거래 채널'] == 'Others')
    ).astype(int)

    df['E,F-입금1000_단말기_기타or윈도우'] = (
        (df['7일 거래내역 중 1천만원 이상 입금 여부'] == 1) &
        (df['거래에 사용한 단말기 OS'].isin(['Others', 'Window']))
    ).astype(int)

    df['E,F-입금1000_채널_기타'] = (
        (df['7일 거래내역 중 1천만원 이상 입금 여부'] == 1) &
        (df['거래 채널'] == 'Others')
    ).astype(int)

    # ✅ [4] 추가 요청: 미거래 단말기 & 고액입금
    df['D-7일 미거래 단말기 & 고액거래'] = (
        (df['7일 거래내역 중 미사용 단말 여부'] == 1) &
        (df['7일 거래내역 중 1천만원 이상 입금 여부'] == 1)
    ).astype(int)

    return df

In [12]:
train_x = add_custom_features(train_x)
test_x = add_custom_features(test_x)

In [13]:
# 컬럼 제거
def drop_columns(df_train, df_test, columns_to_drop):
    # 실제 존재하는 컬럼만 필터링 (오타나 누락 방지)
    columns_in_train = [col for col in columns_to_drop if col in df_train.columns]
    columns_in_test = [col for col in columns_to_drop if col in df_test.columns]

    # 공통으로 제거할 컬럼만 선택
    columns_common = list(set(columns_in_train) & set(columns_in_test))

    print(f"제거되는 컬럼 목록: {columns_common}")

    df_train_dropped = df_train.drop(columns=columns_common)
    df_test_dropped = df_test.drop(columns=columns_common)

    return df_train_dropped, df_test_dropped

In [14]:
columns_to_remove = [
    '고객명',
    '주민번호',
    '수취인 계좌번호',
    '거래에 사용한 단말기 MAC주소',
    '암호화된 계좌번호',
    '거래에 사용한 단말기 IP주소',
    '타계좌 여부',
    '거래 발생 위치',
]

train_x, test_x = drop_columns(train_x, test_x, columns_to_remove)


제거되는 컬럼 목록: ['고객명', '주민번호', '타계좌 여부', '수취인 계좌번호', '거래에 사용한 단말기 IP주소', '암호화된 계좌번호', '거래에 사용한 단말기 MAC주소', '거래 발생 위치']


In [15]:
le_subclass = LabelEncoder()
train_y_encoded = le_subclass.fit_transform(train_y)

# 변환된 레이블 확인
for i, label in enumerate(le_subclass.classes_):
    print(f"원래 레이블: {label}, 변환된 숫자: {i}")

원래 레이블: a, 변환된 숫자: 0
원래 레이블: b, 변환된 숫자: 1
원래 레이블: c, 변환된 숫자: 2
원래 레이블: d, 변환된 숫자: 3
원래 레이블: e, 변환된 숫자: 4
원래 레이블: f, 변환된 숫자: 5
원래 레이블: g, 변환된 숫자: 6
원래 레이블: h, 변환된 숫자: 7
원래 레이블: i, 변환된 숫자: 8
원래 레이블: j, 변환된 숫자: 9
원래 레이블: k, 변환된 숫자: 10
원래 레이블: l, 변환된 숫자: 11
원래 레이블: m, 변환된 숫자: 12


In [16]:
from sklearn.preprocessing import OrdinalEncoder
from pandas.api.types import is_object_dtype, is_categorical_dtype, is_datetime64_any_dtype

# ✅ Step 1. 시간 차이 수치형으로 변환
for df in [train_x, test_x]:
    if '직전 거래와의 시간 차이' in df.columns:
        df['직전 거래와의 시간 차이'] = pd.to_timedelta(df['직전 거래와의 시간 차이'], errors='coerce').dt.total_seconds()

# ✅ Step 2. 범주형 컬럼 선택 (datetime 완전 제외)
categorical_columns = [
    col for col in train_x.columns
    if (is_object_dtype(train_x[col]) or is_categorical_dtype(train_x[col]))
    and not is_datetime64_any_dtype(train_x[col])
    and not pd.api.types.is_datetime64_ns_dtype(train_x[col])
]

# ✅ Step 3. OrdinalEncoder 정의
ordinal_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

# ✅ Step 4. 학습 데이터 인코딩
train_x_encoded = train_x.copy()
train_x_encoded[categorical_columns] = ordinal_encoder.fit_transform(train_x[categorical_columns].astype(str))

# ✅ Step 5. 테스트 데이터 인코딩 (문자열로 통일)
test_x_encoded = test_x.copy()
test_x_encoded[categorical_columns] = ordinal_encoder.transform(test_x[categorical_columns].astype(str))

# ✅ Step 6. 컬럼 순서와 dtype 맞추기
feature_order = train_x_encoded.columns.tolist()
test_x_encoded = test_x_encoded[feature_order]

for col in feature_order:
    test_x_encoded[col] = test_x_encoded[col].astype(train_x_encoded[col].dtype)


In [17]:
# numeric_cols = train_x_encoded.select_dtypes(include=['int64', 'float64']).columns.difference(['target'])
numeric_cols = ['거래 전 잔액',
                '거래 후 잔액',
                '1일 거래 한도',
                '1일 거래 한도 잔여액',
                '1개월 거래내역 중 최대 이체(출금) 금액',
                '1개월 거래내역 이체(출금) 금액 표준편차(중앙값)',
                '1개월 새벽 거래내역 중 최대 이체(출금) 금액',
                '1개월 새벽 거래내역 이체(출금) 금액 표준편처(중앙값)',
                '이체 금액',
                '직전 거래 발생지와의 거리 차이',
                '고객 나이',
                'Amount_vs_Monthly_Max',
                'Amount_Per_Limit_Ratio'

                ]

In [18]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import FactorAnalysis
import pandas as pd

# ✅ 1. 수치형 변수만 선택 (target 제외)
# numeric_cols = train_x_encoded.select_dtypes(include=['int64', 'float64']).columns.difference(['target']).tolist()

# ✅ StandardScaler 정의
scaler = StandardScaler()

# ✅ 스케일링 적용
train_scaled_values = scaler.fit_transform(train_x_encoded[numeric_cols])
test_scaled_values = scaler.transform(test_x_encoded[numeric_cols])

# ✅ 스케일된 결과를 DataFrame으로 변환
train_scaled_df = pd.DataFrame(train_scaled_values, columns=numeric_cols, index=train_x_encoded.index)
test_scaled_df = pd.DataFrame(test_scaled_values, columns=numeric_cols, index=test_x_encoded.index)

# ✅ 최종 스케일링 결과를 원본 순서대로 조합
train_x_scaled_full = pd.DataFrame(index=train_x_encoded.index)
test_x_scaled_full = pd.DataFrame(index=test_x_encoded.index)

for col in train_x_encoded.columns:
    if col in numeric_cols:
        train_x_scaled_full[col] = train_scaled_df[col]
        test_x_scaled_full[col] = test_scaled_df[col]
    else:
        train_x_scaled_full[col] = train_x_encoded[col]
        test_x_scaled_full[col] = test_x_encoded[col]



In [19]:
def clean_column_names(df):
    df.columns = (
        df.columns
        .str.replace(r"[^\w\d_]+", "_", regex=True)  # 한글은 유지, 특수문자 제거
        .str.strip("_")  # 양쪽 밑줄 제거
    )
    return df


In [20]:
train_x_scaled_full = clean_column_names(train_x_scaled_full)
test_x_scaled_full.columns = train_x_scaled_full.columns  # 순서와 이름 통일

In [22]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 6.9 MB/s eta 0:00:00


In [23]:
from catboost import CatBoostClassifier

cat_model = CatBoostClassifier(
    iterations=300,
    learning_rate=0.05,
    depth=6,
    task_type='CPU',     # ✅ 핵심
    random_seed=42,
    loss_function='MultiClass',
    verbose=100
)


In [24]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    use_label_encoder=False,
    eval_metric='mlogloss'
)


In [25]:
from lightgbm import LGBMClassifier

lgb_model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42
)


In [26]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression

stack_model = StackingClassifier(
    estimators=[
        ('cat', cat_model),
        ('xgb', xgb_model),
        ('lgb', lgb_model)
    ],
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5,
    n_jobs=-1
)


In [27]:
import time

start = time.time()
stack_model.fit(train_x_scaled_full, train_y_encoded)
print("총 학습 시간:", round(time.time() - start, 2), "초")
predictions = stack_model.predict(test_x_scaled_full)

# 🎯 예측 결과 라벨 복원
predictions_label = le_subclass.inverse_transform(predictions.ravel().astype(int))

총 학습 시간: 2817.64 초


In [28]:
# 칼럼명이 영어 컬럼으로 일치해야 코드가 돌아감
train = pd.read_csv("./data/train.csv")
a = train['Time_difference']
train = train.drop(['ID','Time_difference'],axis = 1)
train['Time_difference'] = a
all_synthetic_data.columns = train.columns
all_synthetic_data.head()

,Customer_Birthyear,Customer_Gender,Customer_personal_identifier,Customer_identification_number,Customer_registration_datetime,Customer_credit_rating,Customer_flag_change_of_authentication_1,Customer_flag_change_of_authentication_2,Customer_flag_change_of_authentication_3,Customer_flag_change_of_authentication_4,...,Last_bank_branch_transaction_datetime,Flag_deposit_more_than_tenMillion,Unused_account_status,Recipient_account_suspend_status,Number_of_transaction_with_the_account,Transaction_history_with_the_account,First_time_iOS_by_vulnerable_user,Fraud_Type,Transaction_resumed_date,Time_difference
0,1966,female,김영호,dhITBu-DbhAPPn,2012-12-10 22:02:43,D,0,1,1,0,...,2019-09-13 00:21:18,0,0,0,0,0,0,m,2030-06-03 14:14:54,0 days 00:01:38
1,1980,male,이재현,VUdWiC-wXhKmwF,2007-03-25 00:29:47,A,1,1,0,1,...,2011-05-23 14:22:38,0,1,1,1,0,0,m,2030-06-03 14:14:54,19 days 10:54:21
2,1991,female,최유진,UWTuyh-pgXndzV,2012-12-10 22:02:43,A,1,1,1,1,...,2010-07-08 02:23:35,0,1,0,0,0,0,m,2014-05-20 21:31:48,2 days 18:41:06
3,1988,male,이민재,BDBAtF-ZmBUHYl,2006-11-24 04:54:51,D,1,1,0,1,...,2004-08-05 00:53:58,0,0,1,0,0,0,m,2039-02-06 11:20:56,2 days 00:06:39
4,1999,female,김영환,UyFsAc-KBfHGXe,2007-12-20 06:37:08,B,1,1,1,0,...,2020-08-07 17:10:02,0,0,0,2,0,0,m,2004-07-10 20:11:30,12 days 20:26:11.086000


In [29]:
# 분류 예측 결과 제출 데이터프레임(DataFrame)
# 분류 예측 결과 데이터프레임 파일명을 반드시 clf_submission.csv 로 지정해야합니다.
clf_submission = pd.read_csv("data/sample_submission.csv")
clf_submission["Fraud_Type"] = predictions_label
clf_submission.head()

,ID,Fraud_Type
0,TEST_000000,m
1,TEST_000001,m
2,TEST_000002,m
3,TEST_000003,m
4,TEST_000004,m


In [32]:
clf_submission["Fraud_Type"].value_counts()

,count
Fraud_Type,
m,109352
a,3201
j,2359
h,1463
k,1237
f,1121
b,703
i,311
g,162


In [30]:
# 합성 데이터 생성 결과 제출 데이터프레임(DataFrame)
# 합성 데이터 생성 결과 데이터프레임 파일명을 반드시 syn_submission.csv 로 지정해야합니다.
all_synthetic_data.head()

,Customer_Birthyear,Customer_Gender,Customer_personal_identifier,Customer_identification_number,Customer_registration_datetime,Customer_credit_rating,Customer_flag_change_of_authentication_1,Customer_flag_change_of_authentication_2,Customer_flag_change_of_authentication_3,Customer_flag_change_of_authentication_4,...,Last_bank_branch_transaction_datetime,Flag_deposit_more_than_tenMillion,Unused_account_status,Recipient_account_suspend_status,Number_of_transaction_with_the_account,Transaction_history_with_the_account,First_time_iOS_by_vulnerable_user,Fraud_Type,Transaction_resumed_date,Time_difference
0,1966,female,김영호,dhITBu-DbhAPPn,2012-12-10 22:02:43,D,0,1,1,0,...,2019-09-13 00:21:18,0,0,0,0,0,0,m,2030-06-03 14:14:54,0 days 00:01:38
1,1980,male,이재현,VUdWiC-wXhKmwF,2007-03-25 00:29:47,A,1,1,0,1,...,2011-05-23 14:22:38,0,1,1,1,0,0,m,2030-06-03 14:14:54,19 days 10:54:21
2,1991,female,최유진,UWTuyh-pgXndzV,2012-12-10 22:02:43,A,1,1,1,1,...,2010-07-08 02:23:35,0,1,0,0,0,0,m,2014-05-20 21:31:48,2 days 18:41:06
3,1988,male,이민재,BDBAtF-ZmBUHYl,2006-11-24 04:54:51,D,1,1,0,1,...,2004-08-05 00:53:58,0,0,1,0,0,0,m,2039-02-06 11:20:56,2 days 00:06:39
4,1999,female,김영환,UyFsAc-KBfHGXe,2007-12-20 06:37:08,B,1,1,1,0,...,2020-08-07 17:10:02,0,0,0,2,0,0,m,2004-07-10 20:11:30,12 days 20:26:11.086000


In [35]:
train_x_encoded['60세 이후 iOS 첫 사용자'].value_counts()

,count
60세 이후 iOS 첫 사용자,
0,132969
1,31


In [31]:
'''
(*) 저장 시 각 파일명을 반드시 확인해주세요.
    1. 분류 예측 결과 데이터프레임 파일명 = clf_submission.csv
    2. 합성 데이터 생성 결과 데이터프레임 파일명 = syn_submission.csv

(*) 제출 파일(zip) 내에 두 개의 데이터프레임이 각각 위의 파일명으로 반드시 존재해야합니다.
(*) 파일명을 일치시키지 않으면 채점이 불가능합니다.
'''

# 폴더 생성 및 작업 디렉토리 변경
os.makedirs('./submission', exist_ok=True)
os.chdir("./submission/")

# CSV 파일로 저장
clf_submission.to_csv('./clf_submission.csv', encoding='UTF-8-sig', index=False)
all_synthetic_data.to_csv('./syn_submission.csv', encoding='UTF-8-sig', index=False)

# ZIP 파일 생성 및 CSV 파일 추가
with zipfile.ZipFile("../baseline_concat_stack.zip", 'w') as submission:
    submission.write('clf_submission.csv')
    submission.write('syn_submission.csv')

print('Done.')

Done.
